# Jaden: confirmation setup and sample-size planning

**Updated October 20:** your teammates' two prediction CSVs are included in **Jaden_Confirmation_Data.zip**. Download the updated notebook and ZIP together. Upload the `.ipynb` to Colab, run section 1, and choose the **compressed ZIP** when prompted.

Run sections **1–6 one at a time on CPU**. Each cell prints below itself. Section 3 shows **8,881 candidates**. Section 6 calculates the provisional sample-size tables. **Stop there for review**; the GPU and confirmation controls remain off. Section 12 can download the calculated outputs.

| Part | Current status |
|---|---|
| Candidate MATH L3–5 pool | 8,881 candidates after MATH-500 and duplicate/quality exclusions |
| Previous team data | Siddhant reports all generated data came from MATH-500; no other inspected source was reported |
| Received predictions | 342 traces / 74 mixed problems, and 227 traces / 59 mixed problems |
| Sample-size calculations | Four illustrative scenarios, computed separately; final design decisions still needed |
| Exact-token workflow | Code and CPU checks prepared; real GPU validation and generation remain pending |

The smaller cohort overlaps the larger one, and its prediction values differ. They are not independent replications and are not pooled. These CSVs remain development data. Based on the team's reported scope, the candidate pool excludes their previous generation data; no candidate-pool model outputs have been generated or evaluated in this workflow.

This update does not edit your GitHub repository or the team's Drive folder. The notebook does not train probes or run intervention experiments.


In [ ]:
#@title 1. Upload the data ZIP and unpack it
from pathlib import Path
import hashlib, importlib.util, io, json, subprocess, sys, zipfile

print("Step 1 started. If Choose Files appears below, select Jaden_Confirmation_Data.zip.", flush=True)
WORK = Path('/content/jaden_confirmation') if Path('/content').is_dir() else Path.cwd() / 'jaden_confirmation'
ZIP_NAME = 'Jaden_Confirmation_Data.zip'
EXPECTED_ZIP_SHA256 = '33951d0639a447047291a5b49af2cdb0a590ad465121317ad68862e68ef39b71'
EXPECTED_FILES = {'README.txt': 'a62b488e4372cd1118f044f35a50df24803990629e48e888d3932e75e2c4d3b1', 'UPDATE_20261020.txt': '43de420db51e26a7b0b6d4103a062ec4822890d26268ef02838ca8bb6759952a', 'cpu_test_log.txt': '495f0c3aff5d9ddf2fc359b3243c285647d1d212bc7877fd0994b91debd71863', 'data/development/e0_6144_preds_cohort227.csv': '9f1de7e2c74eee8be5e7ada5620add48209df8018b2f2d0d4b478c962c24ee84', 'data/development/e0_6144_preds_pop342.csv': '14e8c9fd9378d8b929da9b8077cdadd88c412345ad99612a783a333dbde6052a', 'data/prepared/candidate_pool.jsonl': '2ae3de181e40fada5b45f3e5f8e78bfb8df99ee1eaf0bc14cb5d141153185c1c', 'data/prepared/exclusions.csv': 'b1e3c5a10a6c45cbd5fbb7eda1d9eef5d296b9369277cce14e15bb39c6dff73a', 'data/prepared/preparation_report.json': 'a2eaeee5ef0f020da515579980cefeb4cc63424cf49dae9b374997b2f5205683', 'data/raw/math500.jsonl': 'a930db7533531003cdb997b8db8cfba3baf6c042a1db1185d59324c95640e6a5', 'data/raw/math_full.jsonl': '83ebae157f5ba1f522abe1e424067e8c661f7df66e225bfec84ff7f132900f3d', 'data/raw/source_download_manifest.json': '2a4db5e236fb96ed5e423dad1c72a710c1ff932f329400857c1052d567dadc32', 'examples/development_oof_schema.csv': '9a54e1d845646601644e5e728ef4db183a81d81667af31c92e3bd83f0b66b6ea', 'input_sources.json': '70d4a628e8013352a44fe4846ed9c13a6c23e6defea94c71cdcff13fc85adb57', 'scripts/exact_token_run.py': 'f49469ceb0b279ef217b66e00ab0f6454f81d8ce5f4a3e168c2fe4493a5b136d', 'scripts/export_pilot_predictions.py': '0abb5c51792c552cdbf9349ec1ffb336bc0d99d1590db89d06e632aa3ed372e5', 'scripts/plan_sample_size.py': 'a08f2f7df7d6cc74fe924a82d6aac5f3a09a319f77492d4efc44daf2b5b67f1a', 'scripts/prepare_problems.py': 'b28e6aadb3d652390722ca13b5cc367d4ee7ae99e885912bfb7a5916d31bd043', 'scripts/wide_prediction_planning.py': 'bd9aeb2a7f6c0390a98410afcebed31995964d0241c0308e35d288309a9195aa', 'team_data_notes_20261020.json': '1715765dec571b98beaa5638454e0886762ffffaae09c61d24006646f6a0060a', 'tests/test_exact_token_run.py': 'efb262c166137ab9239e8e06b6e19d2b8384ba20c18122d12805709fa5cad741', 'tests/test_export_pilot.py': '03d10fa1e1698e361a4e6758cdff224e0af1c7461ec879188d26b50448a9e2ce', 'tests/test_prepare_problems.py': '91c03cb22b6c4c74c6f87b0343a56fbee387d285d51650778d91b4f38d88818d', 'tests/test_sample_size.py': '35ad0d975851a0615b33bafce434cf17754f08158144dcdea55488aa808f2b3b', 'tests/test_wide_prediction_planning.py': 'ea5c7b27c308099ab515ca14cac551dedec3a728e8c5014952579cb2edae5336', 'validation_status.json': 'db38bad0200b157d6157ff25cf35f226bfa79396e80a8438d8a36e4f64e0bc01'}

zip_path = Path.cwd() / ZIP_NAME
if zip_path.is_file() and hashlib.sha256(zip_path.read_bytes()).hexdigest() == EXPECTED_ZIP_SHA256:
    zip_bytes = zip_path.read_bytes()
    print('Using the data ZIP already uploaded to this runtime.', flush=True)
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Place Jaden_Confirmation_Data.zip beside this notebook before running section 1.') from exc
    print('Waiting for you to choose the ZIP file; this is not a calculation.', flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly one file: Jaden_Confirmation_Data.zip. Then rerun this cell.')
    zip_bytes = bytes(next(iter(uploaded.values())))

if hashlib.sha256(zip_bytes).hexdigest() != EXPECTED_ZIP_SHA256:
    raise ValueError('This ZIP does not match the notebook. Download its accompanying Jaden_Confirmation_Data.zip.')
print('Upload verified. Checking files...', flush=True)
# Validate all files before writing; preserve existing outputs and edited files.
resolved_work = WORK.resolve()
if WORK.is_symlink():
    raise ValueError('WORK must be a regular directory, not a symbolic link.')
ready = []
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    names = archive.namelist()
    if len(names) != len(EXPECTED_FILES) or set(names) != set(EXPECTED_FILES):
        raise ValueError('The ZIP has unexpected or duplicate files.')
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or '..' in rel.parts:
            raise ValueError('Unsafe bundle path')
        raw = archive.read(name)
        if hashlib.sha256(raw).hexdigest() != EXPECTED_FILES[name]:
            raise ValueError(f'File verification failed: {name}')
        target = WORK / rel
        if not target.resolve().is_relative_to(resolved_work):
            raise ValueError(f'File would leave the working directory: {name}')
        for part in [target, *target.parents]:
            if part == WORK.parent:
                break
            if part.is_symlink():
                raise ValueError(f'Symbolic links are not allowed inside WORK: {part}')
        if target.exists() and (not target.is_file() or target.read_bytes() != raw):
            raise ValueError(f'{target} was edited; use a new WORK directory to preserve it')
        ready.append((target, raw))
WORK.mkdir(parents=True, exist_ok=True)
for target, raw in ready:
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(raw)
if str(WORK / 'scripts') not in sys.path:
    sys.path.insert(0, str(WORK / 'scripts'))
print('Ready:', WORK, flush=True)
print('Section 1 finished. Now run section 2.', flush=True)
print('Updated bundle includes both development CSVs and the planning helper. Continue with section 2.')


In [ ]:
#@title 2. Install CPU dependencies only if missing, then run the checks
needed = [p for p in ['numpy', 'pandas', 'scipy'] if importlib.util.find_spec(p) is None]
if needed:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *needed], check=True)
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value,'to_string') else value)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(WORK/'tests')], check=True)
print('These are CPU/synthetic checks, not a completed GPU experiment.')


## Check the candidate problems
The included source revisions and file hashes are recorded in `data/raw/source_download_manifest.json`.
The full MATH mirror combines the original collection in one split called `train`; this notebook does not claim a test-only source.
All 500 MATH-500 problems were checked for exclusion; **367 overlap the Level 3–5 pool**.
NFKC/whitespace matching catches formatting variants, but does not establish absence of paraphrases or model pretraining exposure.


In [ ]:
#@title 3. View the prepared pool
report = json.loads((WORK/'data/prepared/preparation_report.json').read_text())
CANDIDATES = WORK/'data/prepared/candidate_pool.jsonl'
assert hashlib.sha256(CANDIDATES.read_bytes()).hexdigest() == report['candidate_file']['sha256']
print('Candidate count:', report['candidate_count'])
display(pd.DataFrame([{'level': k, 'problems': v} for k,v in report['candidate_levels'].items()]))
display(pd.DataFrame([{'reason': k, 'rows_removed': v} for k,v in report['excluded_by_reason'].items()]))


In [ ]:
#@title 4. Record the team's exclusions; add others only if needed
team_notes = json.loads((WORK/'team_data_notes_20261020.json').read_text())
print("Team update:", team_notes['team_reported_scope'])
print("Existing pool:", team_notes['exclusion_status'])
# Optional: add problem/question-text exports if anyone reports additional inspected data.
# Bare numeric IDs without their source mapping are not enough.
INSPECTED_EXPORTS = []
from prepare_problems import prepare_pool
if INSPECTED_EXPORTS:
    extra_report = prepare_pool(WORK/'data/raw/math_full.jsonl', WORK/'data/raw/math500.jsonl',
                                [Path(p) for p in INSPECTED_EXPORTS], WORK/'outputs/updated_pool', 20261018)
    CANDIDATES = WORK/'outputs/updated_pool/candidate_pool.jsonl'
    print('Updated candidate count:', extra_report['candidate_count'])


## Check the received files, then calculate planning scenarios

The original exports use `y_wrong` and scores oriented toward an incorrect answer. The converter flips **both the label and the scores** to the helper's correctness convention; this preserves AUROC.

The CSVs have no fold, seed, configuration hash or checkpoint column. Their names/message identify 6,144 tokens. Their structure is consistent with the supplied code, but the actual exports' out-of-fold and feature-selection provenance is not independently verified.

Section 6 uses **L20_hgb_topk50** against each of **base_lr_len** and **base_lr_tfidf**, separately in each population. These are four illustrative scenarios, not a selected winning method or an approved primary comparison. In the supplied source, `base_lr_len` measures observed prefix tokens and paragraphs; `hgb` is a gradient-boosted ensemble, not a single tree.


In [ ]:
#@title 5. Inspect the two bundled development prediction files
from wide_prediction_planning import convert_wide
DEVELOPMENT_FILES = {
    'pop342': WORK/'data/development/e0_6144_preds_pop342.csv',
    'cohort227': WORK/'data/development/e0_6144_preds_cohort227.csv',
}
data_check = []
for name, path in DEVELOPMENT_FILES.items():
    frame = pd.read_csv(path, dtype={'problem_id':str,'trace_id':str})
    checked, conversion_note = convert_wide(frame, 'L20_hgb_topk50', 'base_lr_len', checkpoint='p6144')
    mixed = checked.groupby('problem_id').y_correct.nunique().eq(2)
    data_check.append({'file':name,'traces':len(checked),'mixed_problems':int(mixed.sum()),
                       'correct':int(checked.y_correct.sum()),'wrong':int((1-checked.y_correct).sum())})
display(pd.DataFrame(data_check))
print('These files contain development predictions. Do not combine their overlapping rows.')


In [ ]:
#@title 6. Calculate provisional sample sizes — CPU only
from wide_prediction_planning import run_bundled_scenarios
scenario_results = run_bundled_scenarios(WORK)
summary = pd.DataFrame(scenario_results['summary'])
sensitivity = pd.DataFrame(scenario_results['sensitivity'])
print('Development results for the four specified scenarios:')
display(summary[['population','probe_column','baseline_column','eligible_mixed_problems',
                 'probe_auroc','baseline_auroc','observed_paired_delta_diagnostic_only']])
print('Illustrative sample sizes: +0.05 or +0.10 AUROC, 80% power, two-sided alpha 0.05.')
preview = sensitivity[sensitivity.useful_auroc_improvement.isin([0.05,0.10]) &
                      sensitivity.target_power.eq(0.80)]
display(preview[['population','baseline_column','useful_auroc_improvement','variance_multiplier',
                 'required_eligible_mixed_problems']])
print('All +0.02/+0.03/+0.05/+0.10 and 80%/90% scenarios were saved to:')
print(WORK/'outputs/development_planning')
print('Counts are NEW independent eligible mixed problems, not traces or total problems to generate.')
print('No total generation target is calculated without an eligibility-yield assumption.')
print('Stop here for mentor/team review. These scenarios do not freeze a confirmation protocol.')
# Illustrative calculations must not satisfy the later confirmation gate.
CONVERTED_PREDICTIONS = None
size_report = None


In [ ]:
#@title 7. Optional: calculate the agreed final planning scenario
# Leave these unchanged until the team settles the design and checks export provenance.
PREDICTIONS_CSV = None  # exact *__correct_oriented.csv inside outputs/development_planning
PRIMARY_CHECKPOINT = 'p6144'
USEFUL_EFFECTS = None  # scientifically useful improvement chosen independently of observed results
ELIGIBLE_FRACTION = None  # expected fraction with both outcomes usable at this checkpoint
TARGET_POWERS = [0.80,0.90]
ALPHA = 0.05
ALTERNATIVE = 'two-sided'
PLANNING_INPUTS_REVIEWED = False
size_report = None
if PREDICTIONS_CSV is not None and USEFUL_EFFECTS is not None:
    if not PLANNING_INPUTS_REVIEWED:
        raise ValueError('Review the primary comparison, effect, population and prediction provenance first.')
    from plan_sample_size import plan_sample_size
    size_report = plan_sample_size(PREDICTIONS_CSV, checkpoint=PRIMARY_CHECKPOINT,
        useful_effects=USEFUL_EFFECTS, powers=TARGET_POWERS, alpha=ALPHA,
        alternative=ALTERNATIVE, eligible_fraction=ELIGIBLE_FRACTION)
    size_report['input_predictions_sha256'] = hashlib.sha256(Path(PREDICTIONS_CSV).read_bytes()).hexdigest()
    out = WORK/'outputs/sample_size'; out.mkdir(parents=True, exist_ok=True)
    (out/'sample_size_plan.json').write_text(json.dumps(size_report, indent=2))
    table = pd.DataFrame(size_report['sensitivity'])
    table.to_csv(out/'sample_size_sensitivity.csv', index=False)
    display(table)
    print(*size_report['warnings'], sep=chr(10))
else:
    print('Final planning scenario not chosen. Section 6 already calculated the illustrative scenarios.')


These are **planning approximations**, not guarantees. They count independent eligible mixed problems, not activation rows or trace pairs. The 1.5× variance case is a sensitivity assumption, not a confidence bound. Shared cross-validation training sets and population changes can make the estimates optimistic.

The received cohorts were already selected by their checkpoint availability. Planning a new population of all trajectories still running at a checkpoint may have a different variance and eligible yield. The fraction of newly generated problems that provides both outcomes at 6,144 tokens is not available from these mixed-only CSVs. Consequently, the **total number of problems to generate is not calculated**.

The illustrative effects are not the observed winning differences and are not approved thresholds. They are assumed true improvements used to plan detection of a difference from zero; the calculation does not aim to prove that a confidence interval exceeds that improvement. Each scenario assumes a single two-sided primary test. Multiple primary methods/checkpoints require a prespecified multiplicity plan. Failure to find a significant difference does not establish equivalence.

Before freezing a new run, settle the exact primary model/baseline, useful improvement, attempt/eligibility policy and practical budget, and confirm the exports' fold/seed/configuration provenance.


## Small GPU smoke test on an existing development problem
This optional test checks whether the code runs on your GPU. It uses one MATH-500 development problem, one attempt and a 1,024-token cap. Its settings differ from the proposed full experiment; **do not include it as confirmation evidence**. It does not validate memory/performance at the full 8,192-token extraction or 32k generation cap.

Use **Runtime → Change runtime type → GPU** in Colab, or your team's existing GPU environment. Keep downloaded outputs; temporary Colab storage can disappear. GPU availability and sufficient runtime are not guaranteed by this notebook.


In [ ]:
#@title 8. Prepare a development smoke-test plan (CPU only)
from exact_token_run import sha_obj, validate_plan, load_inputs
MODEL_REVISION='ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562'
known_dev=[json.loads(line) for line in (WORK/'data/raw/math500.jsonl').read_text().splitlines()]
dev=next(r for r in known_dev if r['level'] in [3,4,5])
pilot_problem={'problem_id':'development-smoke-'+hashlib.sha256(dev['problem'].encode()).hexdigest(),
               'problem':dev['problem'],'gold':str(dev['answer'])}
plans=WORK/'plans';plans.mkdir(parents=True,exist_ok=True)
PILOT_PROBLEMS=plans/'pilot_problem.jsonl'
PILOT_PROBLEMS.write_text(json.dumps(pilot_problem)+chr(10))
pilot_plan={'schema_version':1,'phase':'pilot','model_id':'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
 'model_revision':MODEL_REVISION,'tokenizer_revision':MODEL_REVISION,
 'selected_problem_ids':[pilot_problem['problem_id']],
 'generation':{'samples_per_problem':1,'temperature':0.7,'top_p':0.95,'max_new_tokens':1024,'base_seed':0},
 'extraction':{'checkpoints':[512]}}
validate_plan(pilot_plan)
PILOT_PLAN=plans/'pilot_plan.json';PILOT_PLAN.write_text(json.dumps(pilot_plan,indent=2))
subprocess.run([sys.executable,str(WORK/'scripts/exact_token_run.py'),'validate',
               '--plan',str(PILOT_PLAN),'--problems',str(PILOT_PROBLEMS)],check=True)


In [ ]:
#@title 9. Optional: install the reference GPU dependencies and run the smoke test
RUN_GPU_SMOKE = False #@param {type:'boolean'}
PILOT_OUTPUT=WORK/'outputs/gpu_smoke'
if RUN_GPU_SMOKE:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Select a GPU runtime before this cell.')
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.2','accelerate'],check=True)
    for command in ['generate','extract']:
        subprocess.run([sys.executable,str(WORK/'scripts/exact_token_run.py'),command,
           '--plan',str(PILOT_PLAN),'--problems',str(PILOT_PROBLEMS),'--output-dir',str(PILOT_OUTPUT)],check=True)
    print('GPU smoke commands completed. Review the saved trace and extraction metadata.')
else:
    print('GPU smoke test not run. Generation/extraction remain unvalidated on real hardware.')


## Freeze a confirmation run only after the analysis decisions are settled
The notebook does not decide the useful effect, choose the winning method on the holdout, or infer the total problem count from your old headline score. Fill the fields below with the agreed protocol.

The checkpoint grid below is a draft from the team memo. All trajectories reaching each checkpoint are extracted, with no requirement that they later reach 8,192. Each state is labeled before/after a detected answer marker or unknown. Marker-based labels are not proof of the true answer or commitment onset.

This reference workflow generates new exact-ID traces and re-extracts their activations. It does not reconstruct missing historical IDs or certify the old cache. Generated answers still need the team's grading and probe evaluation afterwards.


In [ ]:
#@title 10. Draft protocol fields — fill these in with the team
N_TOTAL_PROBLEMS = None       # fixed total, supported by the sample-size/yield analysis
PRIMARY_COMPARISON = ''       # exact probe, baseline, fitting/selection rules and primary checkpoint
TRAIN_TEST_PROTOCOL = ''      # e.g. frozen dev-trained monitors evaluated once on new problems
SIZE_JUSTIFICATION = ''       # chosen effect/power/variance/yield row; budget or precision trade-off
EXCLUSION_POLICY = ''         # truncation, answer-stage and checkpoint-eligibility rules
OTHER_INSPECTED_EXCLUSIONS_CHECKED = False
PROTOCOL_REVIEWED = False
SAMPLES_PER_PROBLEM = 6       # draft historical setting; freeze before confirmation
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 32768
CHECKPOINTS = [512,2048,4096,6144,8192]
BASE_SEED = 0
FREEZE_CONFIRMATION = False #@param {type:'boolean'}
CONFIRMATION_PLAN=WORK/'plans/confirmation_plan.json'
PROTOCOL_PATH=WORK/'plans/confirmation_protocol.json'
if FREEZE_CONFIRMATION:
    if not (PROTOCOL_REVIEWED and OTHER_INSPECTED_EXCLUSIONS_CHECKED):
        raise ValueError('Complete the protocol and exclusion review first.')
    if size_report is None or ELIGIBLE_FRACTION is None:
        raise ValueError('Run the size analysis and justify total-problem yield before freezing.')
    if type(N_TOTAL_PROBLEMS) is not int or N_TOTAL_PROBLEMS <= 0:
        raise ValueError('Supply the agreed total problem count.')
    if not all([PRIMARY_COMPARISON.strip(),TRAIN_TEST_PROTOCOL.strip(),SIZE_JUSTIFICATION.strip(),EXCLUSION_POLICY.strip()]):
        raise ValueError('Describe the comparison, training/test plan, exclusions and size justification.')
    candidate_rows=[json.loads(line) for line in CANDIDATES.read_text().splitlines()]
    if N_TOTAL_PROBLEMS > len(candidate_rows):
        raise ValueError('Requested more problems than the available pool; revise the design with the team.')
    chosen=candidate_rows[:N_TOTAL_PROBLEMS]  # ordering was randomized before seeing any new outcomes
    confirmation_plan={'schema_version':1,'phase':'confirmation',
       'model_id':'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
       'model_revision':MODEL_REVISION,'tokenizer_revision':MODEL_REVISION,
       'selected_problem_ids':[r['problem_id'] for r in chosen],
       'generation':{'samples_per_problem':SAMPLES_PER_PROBLEM,'temperature':TEMPERATURE,
                     'top_p':TOP_P,'max_new_tokens':MAX_NEW_TOKENS,'base_seed':BASE_SEED},
       'extraction':{'checkpoints':CHECKPOINTS}}
    validate_plan(confirmation_plan)
    protocol={'primary_comparison':PRIMARY_COMPARISON,'training_test_protocol':TRAIN_TEST_PROTOCOL,
       'size_justification':SIZE_JUSTIFICATION,'exclusion_policy':EXCLUSION_POLICY,
       'sample_size_report_sha256':hashlib.sha256((WORK/'outputs/sample_size/sample_size_plan.json').read_bytes()).hexdigest(),
       'run_plan_sha256':sha_obj(confirmation_plan),'candidate_pool_sha256':hashlib.sha256(CANDIDATES.read_bytes()).hexdigest(),
       'planned_total_problems':N_TOTAL_PROBLEMS,'other_inspected_exclusions_checked':True}
    for path,obj in [(CONFIRMATION_PLAN,confirmation_plan),(PROTOCOL_PATH,protocol)]:
        if path.exists() and json.loads(path.read_text()) != obj:
            raise ValueError('A frozen file differs. Preserve the earlier run and create a new project directory.')
        path.write_text(json.dumps(obj,indent=2))
    print('Plan saved. Maximum generated-token budget (not expected cost):',N_TOTAL_PROBLEMS*SAMPLES_PER_PROBLEM*MAX_NEW_TOKENS)
    print('Frozen settings:',CONFIRMATION_PLAN)
else:
    print('No confirmation sample has been selected or frozen.')


In [ ]:
#@title 11. Full confirmation run — GPU needed; disabled by default
RUN_CONFIRMATION = False #@param {type:'boolean'}
CONFIRMATION_OUTPUT=WORK/'outputs/confirmation_run'
if RUN_CONFIRMATION:
    if not CONFIRMATION_PLAN.exists() or not PROTOCOL_PATH.exists():
        raise ValueError('Freeze the complete plan before running.')
    protocol=json.loads(PROTOCOL_PATH.read_text())
    plan=json.loads(CONFIRMATION_PLAN.read_text())
    if sha_obj(plan) != protocol['run_plan_sha256'] or hashlib.sha256(CANDIDATES.read_bytes()).hexdigest()!=protocol['candidate_pool_sha256']:
        raise ValueError('Frozen plan or candidate pool changed.')
    import torch
    if not torch.cuda.is_available():raise RuntimeError('A CUDA GPU runtime is required.')
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.2','accelerate'],check=True)
    for command in ['generate','extract']:
        args=[sys.executable,str(WORK/'scripts/exact_token_run.py'),command,
              '--plan',str(CONFIRMATION_PLAN),'--problems',str(CANDIDATES),'--output-dir',str(CONFIRMATION_OUTPUT)]
        if command=='generate':args.append('--confirm-confirmation-run')
        subprocess.run(args,check=True)
    print('Generation and extraction commands completed. Grading/probe confirmation is still a separate team task.')
else:
    print('No full generation or extraction launched.')


In [ ]:
#@title 12. Download your outputs before ending the Colab session
DOWNLOAD_OUTPUTS = False #@param {type:'boolean'}
if DOWNLOAD_OUTPUTS:
    import zipfile
    export=WORK.parent/'Jaden_Confirmation_Outputs.zip'
    with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as z:
        for folder in [WORK/'outputs',WORK/'plans']:
            if folder.exists():
                for f in folder.rglob('*'):
                    if f.is_file():z.write(f,str(f.relative_to(WORK)))
        z.write(WORK/'validation_status.json','preparation_validation_status.json')
    from google.colab import files
    files.download(str(export))
else:
    print('Prepared files live in',WORK,'— save actual run outputs before runtime deletion.')


## What is ready to share

The new `outputs/development_planning` folder contains the development comparison summary, sample-size sensitivity table, converted inputs and calculation metadata. The notebook's default run leaves the final protocol, confirmation sample selection and GPU runs disabled.

The candidate-pool report and Siddhant's scope clarification support excluding MATH-500. If anyone has inspected other candidate problems, add those exclusions before freezing the sample.

**Still pending:** agreement on the primary comparison and useful effect, confirmation of prediction provenance and eligibility assumptions, a final sample/compute target, and real GPU validation followed by the approved run. The development calculations do not confirm an early-warning signal.

Source datasets and model revisions are recorded in the bundled source manifest. The two received CSVs are included unchanged and hashed in `team_data_notes_20261020.json`.
